# 02 - Monte Carlo and Convergence

Risk-neutral Monte Carlo for vanilla, Asian, and barrier options. Every estimate carries
its standard error, and the convergence of that error like $1/\sqrt{M}$ is shown directly.
Barrier prices are validated through in-out parity.


## Setup

In [ ]:
!pip install -q numpy scipy matplotlib pandas

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "your-username"   # <-- edit this
REPO = "derivatives-pricing"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)

!pip install -q -e .
src = os.path.abspath("src")
if src not in sys.path:
    sys.path.insert(0, src)
import derivpricing
print("derivpricing", derivpricing.__version__)

## Vanilla European against Black-Scholes

The Monte Carlo price should sit within a couple of standard errors of the closed form.


In [ ]:
from derivpricing.analytic import bs_price
from derivpricing.montecarlo import mc_european

S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2
bs = bs_price(S0, K, T, r, sigma, "call")
price, se = mc_european(S0, K, T, r, sigma, M=200_000, option_type="call", seed=1)
print(f"Black-Scholes {bs:.4f}")
print(f"Monte Carlo   {price:.4f}  (SE {se:.4f})  ->  {abs(price-bs)/se:.2f} standard errors away")

## Convergence of the standard error

As the sample size $M$ grows, the estimate tightens onto the analytic price and the
standard error falls like $1/\sqrt{M}$. The right panel shows that the standard error
tracks a straight line against $1/\sqrt{M}$ on log axes.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from derivpricing.montecarlo import mc_convergence

sizes, prices, ses = mc_convergence(
    mc_european, np.logspace(2.5, 6, 12).astype(int),
    S0=S0, K=K, T=T, r=r, sigma=sigma, option_type="call", seed=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].errorbar(sizes, prices, yerr=1.96*ses, fmt="o-", capsize=3, label="MC price (95% band)")
ax[0].axhline(bs, color="black", ls="--", label=f"Black-Scholes {bs:.4f}")
ax[0].set_xscale("log"); ax[0].set_xlabel("samples M"); ax[0].set_ylabel("price"); ax[0].legend()
ax[0].set_title("Estimate tightening with M")
ax[1].loglog(sizes, ses, "o-", label="standard error")
ax[1].loglog(sizes, ses[0]*np.sqrt(sizes[0])/np.sqrt(sizes), "k--", label="reference 1/sqrt(M)")
ax[1].set_xlabel("samples M"); ax[1].set_ylabel("standard error"); ax[1].legend()
ax[1].set_title("Standard error vs 1/sqrt(M)")
plt.tight_layout(); plt.show()

## Arithmetic-average Asian option

A path-dependent payoff on the average price, which has no simple closed form and sits
below the comparable vanilla because averaging dampens the terminal dispersion.


In [ ]:
from derivpricing.montecarlo import mc_asian
a_price, a_se = mc_asian(S0, K, T, r, sigma, N=50, M=100_000, option_type="call", seed=1)
print(f"Asian call {a_price:.4f}  (SE {a_se:.4f})   vanilla call {bs:.4f}")

## Barrier options and in-out parity

A knock-out and its matching knock-in, priced on the same paths, sum to the vanilla option.
This parity holds exactly at the shared monitoring grid, which validates the barrier pricer.


In [ ]:
from derivpricing.montecarlo import mc_barrier, gbm_paths
H, N, M, seed = 90.0, 252, 200_000, 7
out, _ = mc_barrier(S0, K, H, T, r, sigma, "down-and-out", N, M, "call", seed=seed)
inn, _ = mc_barrier(S0, K, H, T, r, sigma, "down-and-in",  N, M, "call", seed=seed)
paths = gbm_paths(S0, T, r, sigma, N, M, seed=seed)
vanilla_paths = float(np.exp(-r*T) * np.maximum(paths[:, -1] - K, 0.0).mean())
print(f"down-and-out {out:.4f}")
print(f"down-and-in  {inn:.4f}")
print(f"sum          {out+inn:.4f}   vanilla on same paths {vanilla_paths:.4f}")

## Recap

The vanilla Monte Carlo agrees with Black-Scholes, the standard error falls like the square
root of the sample size, and the barrier pricer satisfies in-out parity. Notebook `03`
turns to the Heston model and calibration.
